# MNIST from Scratch — Every Line Explained

**Course:** ML, Deep Learning & Computer Vision — Phase 3  
**Goal:** Train a neural network to recognise handwritten digits (0–9)  
**Prerequisites:** Phase 3 theory + PyTorch deep dive notebooks  

---

MNIST is the "Hello World" of deep learning — 70,000 grayscale images of handwritten digits, each 28×28 pixels.  
We'll build a feedforward network that achieves >97% accuracy, understanding **every single line** of code.

### What you'll build

```
Input (784 pixels) → FC(256, ReLU) → Dropout(0.2) → FC(128, ReLU) → Dropout(0.2) → FC(10) → Softmax → Prediction
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import time

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__} | Device: {device}")

---
## 1. Load and explore the data

MNIST is built into `torchvision`. The `transforms` pipeline converts images to tensors and normalises them.

In [ ]:
# === DATA PIPELINE ===
# transforms.Compose chains multiple preprocessing steps
transform = transforms.Compose([
    transforms.ToTensor(),        # PIL image → tensor, scales [0,255] → [0,1]
    transforms.Normalize(         # normalise: (x - mean) / std
        mean=(0.1307,),           # MNIST mean (precomputed)
        std=(0.3081,)             # MNIST std (precomputed)
    ),
])
# After this pipeline:
#   - Each image is a tensor of shape (1, 28, 28) — 1 channel, 28×28 pixels
#   - Pixel values have mean≈0 and std≈1 (good for training)

# Download and load MNIST
train_dataset = datasets.MNIST(
    root='./data',          # where to store/find data
    train=True,             # training set (60,000 images)
    download=True,          # download if not found
    transform=transform,    # apply preprocessing
)

test_dataset = datasets.MNIST(
    root='./data',
    train=False,            # test set (10,000 images)
    download=True,
    transform=transform,
)

print(f"Training samples: {len(train_dataset):,}")
print(f"Test samples:     {len(test_dataset):,}")

# Inspect one sample
image, label = train_dataset[0]
print(f"\nImage shape:  {image.shape}")     # (1, 28, 28)
print(f"Image dtype:  {image.dtype}")        # float32
print(f"Pixel range:  [{image.min():.2f}, {image.max():.2f}]")  # normalised
print(f"Label:        {label}")

In [ ]:
# === VISUALISE SAMPLES ===
fig, axes = plt.subplots(2, 10, figsize=(15, 3.5))
fig.suptitle('MNIST — handwritten digits', fontsize=14, fontweight='bold', y=1.02)

for i in range(20):
    ax = axes[i // 10, i % 10]
    image, label = train_dataset[i]
    # image is (1, 28, 28) — squeeze to (28, 28) for plotting
    ax.imshow(image.squeeze(), cmap='gray')
    ax.set_title(str(label), fontsize=11, color='#378ADD')
    ax.axis('off')

plt.tight_layout()
plt.show()

# Class distribution
labels = [train_dataset[i][1] for i in range(len(train_dataset))]
unique, counts = np.unique(labels, return_counts=True)
print("Class distribution:")
for digit, count in zip(unique, counts):
    print(f"  Digit {digit}: {count:,} samples ({count/len(labels):.1%})")

In [ ]:
# === DATALOADERS ===
# DataLoader wraps the dataset and handles batching, shuffling, and parallel loading

BATCH_SIZE = 128  # process 128 images at a time

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,        # IMPORTANT: shuffle training data each epoch
    num_workers=2,       # load data in parallel (speeds up training)
    pin_memory=True,     # faster CPU→GPU transfer
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,       # larger batch for eval (no gradients stored)
    shuffle=False,        # don't shuffle test data
    num_workers=2,
    pin_memory=True,
)

# Check one batch
batch_X, batch_y = next(iter(train_loader))
print(f"Batch X shape: {batch_X.shape}")  # (128, 1, 28, 28)
print(f"Batch y shape: {batch_y.shape}")  # (128,)
print(f"Batches per epoch: {len(train_loader)}")

---
## 2. Build the model

Architecture:
- **Flatten** 28×28 image → 784-dim vector
- **FC layer 1:** 784 → 256 neurons, ReLU, Dropout(0.2)
- **FC layer 2:** 256 → 128 neurons, ReLU, Dropout(0.2)
- **FC layer 3:** 128 → 10 (one score per digit)

Total: ~235K parameters

In [ ]:
class MNISTClassifier(nn.Module):
    """Feedforward network for MNIST digit classification."""
    
    def __init__(self):
        super().__init__()
        
        # nn.Flatten() converts (batch, 1, 28, 28) → (batch, 784)
        self.flatten = nn.Flatten()
        
        # Hidden layers
        self.fc1 = nn.Linear(784, 256)    # 784*256 + 256 = 200,960 params
        self.fc2 = nn.Linear(256, 128)    # 256*128 + 128 = 32,896 params
        self.fc3 = nn.Linear(128, 10)     # 128*10 + 10 = 1,290 params
        
        # Regularisation
        self.dropout = nn.Dropout(0.2)    # randomly zero 20% of neurons
        
        # Batch normalisation (stabilises training)
        self.bn1 = nn.BatchNorm1d(256)
        self.bn2 = nn.BatchNorm1d(128)
    
    def forward(self, x):
        # x shape: (batch, 1, 28, 28)
        x = self.flatten(x)        # (batch, 784)
        
        x = self.fc1(x)            # (batch, 256)
        x = self.bn1(x)            # batch norm
        x = F.relu(x)              # activation
        x = self.dropout(x)        # regularisation
        
        x = self.fc2(x)            # (batch, 128)
        x = self.bn2(x)            # batch norm
        x = F.relu(x)              # activation
        x = self.dropout(x)        # regularisation
        
        x = self.fc3(x)            # (batch, 10) — raw logits
        return x                    # DON'T apply softmax here — CrossEntropyLoss does it

# Create and inspect
model = MNISTClassifier().to(device)
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

# Verify forward pass works
with torch.no_grad():
    test_output = model(batch_X.to(device))
    print(f"\nTest forward pass: input {batch_X.shape} → output {test_output.shape}")

---
## 3. Train the model

Components:
- **Loss:** `CrossEntropyLoss` — combines softmax + negative log likelihood
- **Optimiser:** Adam with lr=0.001
- **Scheduler:** ReduceLROnPlateau — reduces lr when validation loss plateaus

In [ ]:
# === TRAINING SETUP ===
criterion = nn.CrossEntropyLoss()  # softmax + CE loss in one
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, verbose=True
)

print("Loss:      CrossEntropyLoss")
print("Optimiser: Adam (lr=0.001)")
print("Scheduler: ReduceLROnPlateau (halve lr if val loss stalls for 3 epochs)")

In [ ]:
# === TRAINING + EVALUATION FUNCTIONS ===

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        
        optimizer.zero_grad()        # clear old gradients
        logits = model(X)            # forward pass
        loss = criterion(logits, y)  # compute loss
        loss.backward()              # compute gradients
        optimizer.step()             # update weights
        
        running_loss += loss.item() * X.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += X.size(0)
    
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        loss = criterion(logits, y)
        
        running_loss += loss.item() * X.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += X.size(0)
    
    return running_loss / total, correct / total

print("Functions defined. Ready to train!")

In [ ]:
# === TRAIN! ===
N_EPOCHS = 15
best_val_acc = 0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print(f"{'Epoch':>5} {'Train Loss':>12} {'Train Acc':>10} {'Val Loss':>10} {'Val Acc':>9} {'LR':>10} {'Time':>6}")
print("-" * 70)

for epoch in range(1, N_EPOCHS + 1):
    t0 = time.time()
    
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)
    
    scheduler.step(val_loss)  # adjust learning rate
    
    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_mnist_model.pth')
    
    elapsed = time.time() - t0
    lr = optimizer.param_groups[0]['lr']
    print(f"{epoch:5d} {train_loss:12.4f} {train_acc:10.2%} {val_loss:10.4f} {val_acc:9.2%} {lr:10.6f} {elapsed:5.1f}s")

print(f"\nBest validation accuracy: {best_val_acc:.2%}")

In [ ]:
# === TRAINING CURVES ===
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
epochs = range(1, N_EPOCHS + 1)

# Loss
axes[0].plot(epochs, history['train_loss'], 'o-', color='#378ADD', linewidth=2, markersize=4, label='Train')
axes[0].plot(epochs, history['val_loss'], 's--', color='#D85A30', linewidth=2, markersize=4, label='Test')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-entropy loss')
axes[0].set_title('Loss', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.2)

# Accuracy
axes[1].plot(epochs, history['train_acc'], 'o-', color='#378ADD', linewidth=2, markersize=4, label='Train')
axes[1].plot(epochs, history['val_acc'], 's--', color='#D85A30', linewidth=2, markersize=4, label='Test')
axes[1].axhline(best_val_acc, color='#1D9E75', linestyle=':', alpha=0.7, label=f'Best={best_val_acc:.2%}')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy', fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.2)
axes[1].set_ylim(0.9, 1.0)

plt.suptitle('MNIST Training Results', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

---
## 4. Evaluate & understand the model

Numbers alone don't tell the full story. Let's look at:
- Confusion matrix (which digits get confused?)
- Per-class accuracy
- Individual predictions (correct and incorrect)
- What the model learned (weight visualisation)

In [ ]:
# === CONFUSION MATRIX ===
model.load_state_dict(torch.load('best_mnist_model.pth', weights_only=True))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for X, y in test_loader:
        logits = model(X.to(device))
        preds = logits.argmax(1).cpu()
        all_preds.append(preds)
        all_labels.append(y)

all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

# Build confusion matrix
from collections import Counter
cm = np.zeros((10, 10), dtype=int)
for true, pred in zip(all_labels, all_preds):
    cm[true, pred] += 1

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap='Blues')
for i in range(10):
    for j in range(10):
        color = 'white' if cm[i, j] > cm.max() * 0.5 else 'black'
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', color=color, fontsize=9)

ax.set_xlabel('Predicted digit', fontsize=12)
ax.set_ylabel('True digit', fontsize=12)
ax.set_title('Confusion Matrix', fontweight='bold', fontsize=14)
ax.set_xticks(range(10)); ax.set_yticks(range(10))
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

# Per-class accuracy
print("\nPer-digit accuracy:")
for digit in range(10):
    correct = cm[digit, digit]
    total = cm[digit].sum()
    acc = correct / total
    bar = '█' * int(acc * 40) + '░' * (40 - int(acc * 40))
    print(f"  Digit {digit}: {bar} {acc:.1%} ({correct}/{total})")

In [ ]:
# === VISUALISE PREDICTIONS ===
# Show correct and incorrect predictions
correct_idx = np.where(all_preds == all_labels)[0]
wrong_idx = np.where(all_preds != all_labels)[0]

fig, axes = plt.subplots(2, 10, figsize=(16, 4))
fig.suptitle('Predictions', fontsize=14, fontweight='bold', y=1.02)

# Top row: correct predictions
axes[0, 0].set_ylabel('Correct', fontsize=11, color='#1D9E75', fontweight='bold')
for i in range(10):
    idx = correct_idx[i]
    img = test_dataset[idx][0].squeeze()
    axes[0, i].imshow(img, cmap='gray')
    axes[0, i].set_title(f'{all_preds[idx]}', color='#1D9E75', fontweight='bold')
    axes[0, i].axis('off')

# Bottom row: wrong predictions
axes[1, 0].set_ylabel('Wrong', fontsize=11, color='#E24B4A', fontweight='bold')
for i in range(min(10, len(wrong_idx))):
    idx = wrong_idx[i]
    img = test_dataset[idx][0].squeeze()
    axes[1, i].imshow(img, cmap='gray')
    axes[1, i].set_title(f'{all_preds[idx]}(✗{all_labels[idx]})', color='#E24B4A', fontsize=9)
    axes[1, i].axis('off')

plt.tight_layout(); plt.show()

print(f"Total: {len(all_labels):,} test images")
print(f"Correct: {len(correct_idx):,} ({len(correct_idx)/len(all_labels):.2%})")
print(f"Wrong:   {len(wrong_idx):,} ({len(wrong_idx)/len(all_labels):.2%})")

In [ ]:
# === VISUALISE FIRST LAYER WEIGHTS ===
# Each neuron in fc1 has 784 weights — reshape to 28×28 to see what "pattern" it looks for

weights = model.fc1.weight.data.cpu().numpy()  # (256, 784)

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
fig.suptitle('First layer weight patterns (what each neuron looks for)', fontsize=13, fontweight='bold', y=1.01)

for i, ax in enumerate(axes.flat):
    w = weights[i].reshape(28, 28)
    ax.imshow(w, cmap='RdBu_r', vmin=-0.3, vmax=0.3)
    ax.axis('off')
    ax.set_title(f'N{i}', fontsize=7)

plt.tight_layout(); plt.show()

print("Red regions = positive weights (neuron activates when these pixels are bright)")
print("Blue regions = negative weights (neuron activates when these pixels are dark)")
print("Some neurons look like edge detectors, others respond to specific digit shapes.")

In [ ]:
# === PREDICTION ON A SINGLE IMAGE ===
# This is what inference looks like in production

def predict_digit(model, image_tensor, device):
    """Predict a single digit and return probabilities."""
    model.eval()
    with torch.no_grad():
        logits = model(image_tensor.unsqueeze(0).to(device))  # add batch dim
        probs = F.softmax(logits, dim=1).cpu().squeeze()
    return probs

# Pick a test image
idx = 42
image, true_label = test_dataset[idx]
probs = predict_digit(model, image, device)
pred_label = probs.argmax().item()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Image
axes[0].imshow(image.squeeze(), cmap='gray')
axes[0].set_title(f'True: {true_label}, Predicted: {pred_label}', fontsize=13, fontweight='bold')
axes[0].axis('off')

# Probability bar chart
colors = ['#1D9E75' if i == pred_label else '#378ADD' for i in range(10)]
bars = axes[1].bar(range(10), probs.numpy(), color=colors)
axes[1].set_xticks(range(10))
axes[1].set_xlabel('Digit'); axes[1].set_ylabel('Probability')
axes[1].set_title('Prediction confidence', fontweight='bold')
axes[1].set_ylim(0, 1)
# Annotate top prediction
axes[1].text(pred_label, probs[pred_label].item() + 0.03,
             f'{probs[pred_label].item():.1%}', ha='center', fontweight='bold', color='#1D9E75')

plt.tight_layout(); plt.show()

---
## 5. Experiments to try

Now that you have a working baseline, try these modifications to build intuition:

| Experiment | What to change | Expected effect |
|-----------|---------------|----------------|
| 1. No dropout | Remove `Dropout` layers | Higher train acc, lower test acc (overfitting) |
| 2. No batch norm | Remove `BatchNorm1d` layers | Slower convergence |
| 3. Deeper network | Add more hidden layers (3–4) | Might improve, or might be harder to train |
| 4. Wider network | Use 512 and 256 neurons | More capacity, might overfit |
| 5. Higher dropout | Use `Dropout(0.5)` | Stronger regularisation, might hurt if too much |
| 6. Lower learning rate | Use `lr=1e-4` | Slower but more stable convergence |
| 7. SGD instead of Adam | `torch.optim.SGD(lr=0.01, momentum=0.9)` | Slower, sometimes better final result |
| 8. No normalisation | Remove `Normalize` from transforms | See how normalisation helps |

For each experiment, compare the training curves against the baseline.

In [ ]:
# Clean up saved model
import os
if os.path.exists('best_mnist_model.pth'):
    os.remove('best_mnist_model.pth')
    print("Cleaned up model file.")

---
## Summary

What we built:
1. **Data pipeline:** `transforms.Compose` → `datasets.MNIST` → `DataLoader`
2. **Model:** `nn.Module` with Linear + BatchNorm + ReLU + Dropout
3. **Training:** `CrossEntropyLoss` + `Adam` + `ReduceLROnPlateau`
4. **Evaluation:** Confusion matrix, per-class accuracy, prediction visualisation
5. **Interpretability:** First-layer weight visualisation

Key numbers:
- ~235K parameters
- ~97–98% accuracy on test set
- Trains in ~2 minutes on CPU, ~30 seconds on GPU

**Next in Phase 4:** We'll switch from fully-connected layers to Convolutional Neural Networks (CNNs), which exploit the 2D spatial structure of images and push MNIST accuracy to >99%.